# MEDI-Zero 피부 질환 분류 모델 학습
**모델**: EfficientNet-B4 (전이학습)  
**클래스**: 15개 피부 질환  
**XAI**: Grad-CAM 시각화  

## 데이터셋 구조 (Kaggle Dataset에 업로드 필요)
```
skin_dataset/
  광선각화증/   *.png
  기저세포암/   *.png
  멜라닌세포모반/ *.png
  보웬병/      *.png
  비립종/      *.png
  사마귀/      *.png
  악성흑색종/   *.png
  지루각화증/   *.png
  편평세포암/   *.png
  표피낭종/    *.png
  피부섬유종/   *.png
  피지샘증식증/  *.png
  혈관종/      *.png
  화농 육아종/  *.png
  흑색점/      *.png
```

In [ ]:
!pip install -q grad-cam timm

In [ ]:
import os
import json
import glob
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms, datasets
from sklearn.metrics import classification_report, confusion_matrix

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── 설정 ──────────────────────────────────────────────
BASE            = '/kaggle/input/datasets/kimminsu0109/skin-dataset'
TRAIN_IMG_DIR   = f'{BASE}/Train'
TRAIN_LABEL_DIR = f'{BASE}/Train_label'
VAL_IMG_DIR     = f'{BASE}/Val'
VAL_LABEL_DIR   = f'{BASE}/Val_label'
OUTPUT_DIR      = '/kaggle/working'
MODEL_PATH      = os.path.join(OUTPUT_DIR, 'skin_efficientnet_b4.pth')
CLASS_MAP_PATH  = os.path.join(OUTPUT_DIR, 'class_to_idx.json')

INPUT_SIZE   = 380
BBOX_PADDING = 20
BATCH_SIZE   = 32
EPOCHS       = 40
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
PATIENCE     = 7

# Train=TS_ / Val=VS_ 둘 다 같은 질환명으로 매핑
PREFIX_TO_KO = {
    'TS_광선각화증':    '광선각화증',
    'TS_기저세포암':    '기저세포암',
    'TS_멜라닌세포모반': '멜라닌세포모반',
    'TS_보웬병':       '보웬병',
    'TS_비립종':       '비립종',
    'TS_사마귀':       '사마귀',
    'TS_악성흑색종':    '악성흑색종',
    'TS_지루각화증':    '지루각화증',
    'TS_편평세포암':    '편평세포암',
    'TS_표피낭종':     '표피낭종',
    'TS_피부섬유종':    '피부섬유종',
    'TS_피지샘증식증':   '피지샘증식증',
    'TS_혈관종':       '혈관종',
    'TS_화농 육아종':   '화농 육아종',
    'TS_흑색점':       '흑색점',
    'VS_광선각화증':    '광선각화증',
    'VS_기저세포암':    '기저세포암',
    'VS_멜라닌세포모반': '멜라닌세포모반',
    'VS_보웬병':       '보웬병',
    'VS_비립종':       '비립종',
    'VS_사마귀':       '사마귀',
    'VS_악성흑색종':    '악성흑색종',
    'VS_지루각화증':    '지루각화증',
    'VS_편평세포암':    '편평세포암',
    'VS_표피낭종':     '표피낭종',
    'VS_피부섬유종':    '피부섬유종',
    'VS_피지샘증식증':   '피지샘증식증',
    'VS_혈관종':       '혈관종',
    'VS_화농 육아종':   '화농 육아종',
    'VS_흑색점':       '흑색점',
}

# 클래스 인덱스는 질환명 기준으로 고정 (TS/VS 무관하게 동일 idx)
KO_CLASSES   = sorted(set(PREFIX_TO_KO.values()))
KO_TO_IDX    = {ko: i for i, ko in enumerate(KO_CLASSES)}
CLASS_TO_IDX = {folder: KO_TO_IDX[ko] for folder, ko in PREFIX_TO_KO.items()}
IDX_TO_KO    = {i: ko for ko, i in KO_TO_IDX.items()}
NUM_CLASSES  = len(KO_CLASSES)

# 폴더 존재 여부 확인
for name, path in [('Train 이미지', TRAIN_IMG_DIR), ('Train 라벨', TRAIN_LABEL_DIR),
                   ('Val 이미지',   VAL_IMG_DIR),   ('Val 라벨',   VAL_LABEL_DIR)]:
    status = '✅' if os.path.isdir(path) else '❌ 없음'
    print(f'{status}  {name}: {path}')
print(f'\n클래스 수: {NUM_CLASSES}')

In [ ]:
# ── 커스텀 Dataset (bbox 크롭 적용) ───────────────────
IMG_EXTS = ('*.png', '*.jpg', '*.jpeg')

class SkinDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.transform = transform
        self.samples   = []

        for folder in sorted(os.listdir(img_dir)):
            if folder not in CLASS_TO_IDX:
                continue
            cls_idx    = CLASS_TO_IDX[folder]
            img_folder = os.path.join(img_dir,   folder)
            lbl_folder = os.path.join(label_dir, folder)

            img_paths = []
            for ext in IMG_EXTS:
                img_paths.extend(glob.glob(os.path.join(img_folder, ext)))

            for img_path in sorted(img_paths):
                stem      = Path(img_path).stem
                json_path = os.path.join(lbl_folder, stem + '.json')
                self.samples.append((
                    img_path,
                    json_path if os.path.exists(json_path) else None,
                    cls_idx,
                ))

    def __len__(self):
        return len(self.samples)

    def _crop_bbox(self, img, json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        bbox = data['annotations'][0].get('bbox', {})
        x, y = bbox.get('xpos', 0), bbox.get('ypos', 0)
        w, h = bbox.get('width', img.width), bbox.get('height', img.height)
        x1 = max(0,          x - BBOX_PADDING)
        y1 = max(0,          y - BBOX_PADDING)
        x2 = min(img.width,  x + w + BBOX_PADDING)
        y2 = min(img.height, y + h + BBOX_PADDING)
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        img_path, json_path, cls_idx = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if json_path:
            try:
                img = self._crop_bbox(img, json_path)
            except Exception:
                pass
        if self.transform:
            img = self.transform(img)
        return img, cls_idx


# ── Transform ─────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE + 20, INPUT_SIZE + 20)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── DataLoader 생성 ────────────────────────────────────
train_dataset = SkinDataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, transform=train_transform)
val_dataset   = SkinDataset(VAL_IMG_DIR,   VAL_LABEL_DIR,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# class_to_idx 저장
with open(CLASS_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump({'folder_to_idx': CLASS_TO_IDX,
               'idx_to_ko':     {str(k): v for k, v in IDX_TO_KO.items()}},
              f, ensure_ascii=False, indent=2)

print(f'Train: {len(train_dataset)}장 | Val: {len(val_dataset)}장')
if len(val_dataset) == 0:
    print('⚠️  Val 데이터가 없습니다. Val 폴더 구조를 확인하세요:')
    for folder in sorted(os.listdir(VAL_IMG_DIR))[:3]:
        files = os.listdir(os.path.join(VAL_IMG_DIR, folder))
        print(f'  {folder}/ → {len(files)}개, 예시: {files[:2]}')

In [ ]:
# ── 클래스 분포 시각화 ─────────────────────────────────
class_counts = pd.Series([full_dataset.classes[label] for _, label in full_dataset.samples])
plt.figure(figsize=(14, 5))
class_counts.value_counts().plot(kind='bar', color='steelblue')
plt.title('클래스별 이미지 수', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

In [ ]:
# ── 모델 정의 ─────────────────────────────────────────
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total_params:,} | 학습 파라미터: {train_params:,}')

In [ ]:
# ── 학습 설정 ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    if len(loader.dataset) == 0:
        raise RuntimeError('Val 데이터셋이 비어있습니다. Val 폴더를 확인해주세요.')
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

In [ ]:
# ── 학습 루프 ─────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
no_improve = 0
UNFREEZE_EPOCH = 10  # 10 에포크 후 백본 전체 파인튜닝

for epoch in range(1, EPOCHS + 1):

    # 백본 언프리즈
    if epoch == UNFREEZE_EPOCH:
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=LR * 0.1, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - UNFREEZE_EPOCH, eta_min=1e-7)
        print(f'[Epoch {epoch}] 백본 언프리즈 - 전체 파인튜닝 시작')

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch [{epoch:02d}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  => Best model saved (val_acc={best_val_acc:.4f})')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\n학습 완료. Best Val Acc: {best_val_acc:.4f}')

In [ ]:
# ── 학습 곡선 시각화 ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'],   label='Val Acc')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── 최종 평가 ─────────────────────────────────────────
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

# class_names: TS_ 접두어 제거된 한국어 이름 사용
print('=== Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# ── 혼동 행렬 시각화 ───────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(16, 13))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Normalized)', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ── 클래스별 정확도 막대그래프 ─────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(14, 5))
bars = plt.bar(class_names, per_class_acc, color='steelblue')
plt.axhline(y=per_class_acc.mean(), color='red', linestyle='--', label=f'Mean: {per_class_acc.mean():.3f}')
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2f}', ha='center', va='bottom', fontsize=9)
plt.title('클래스별 정확도', fontsize=14)
plt.ylim(0, 1.1)
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_accuracy.png'), dpi=150)
plt.show()

In [ ]:
# ── Grad-CAM 샘플 시각화 ───────────────────────────────
# 각 클래스에서 1장씩 샘플링하여 Grad-CAM 오버레이 생성
model.eval()
target_layers = [model.features[-1]]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

# 검증셋에서 클래스별 첫 번째 샘플 추출
class_samples = {}
for path, label in val_dataset.samples:
    if label not in class_samples:
        class_samples[label] = path
    if len(class_samples) == NUM_CLASSES:
        break

inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

for cls_idx in range(NUM_CLASSES):
    if cls_idx not in class_samples:
        continue
    img_path = class_samples[cls_idx]
    pil_img = Image.open(img_path).convert('RGB')
    tensor = val_transform(pil_img).unsqueeze(0).to(DEVICE)

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=tensor,
                            targets=[ClassifierOutputTarget(cls_idx)])[0]

    rgb_img = np.array(pil_img.resize((INPUT_SIZE, INPUT_SIZE)), dtype=np.float32) / 255.0
    overlay = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    axes[cls_idx].imshow(overlay)
    axes[cls_idx].set_title(class_names[cls_idx], fontsize=10)
    axes[cls_idx].axis('off')

plt.suptitle('Grad-CAM 시각화 (클래스별 병변 위치)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM 저장 완료')

In [ ]:
# ── 출력 파일 확인 ─────────────────────────────────────
print('=== 생성된 파일 ===')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:40s}  {size/1024:.1f} KB')